# Crossdating tree-ring data with dplPy

A tour of dplPy's crossdating tools — the checks that confirm every ring is
assigned to its correct calendar year. We build a leave-one-out master
chronology and correlate each series against it, flag segments that don't match,
locate dating shifts, date a floating series, and produce a COFECHA-style report.

Uses the sample files in `../tests/data/rwl/`, so it runs as-is from the
`notebooks/` folder of the repository.

In [ ]:
import dplpy as dpl
import matplotlib.pyplot as plt
import pandas as pd

dpl.__version__

In [ ]:
RWL = "../tests/data/rwl/"   # sample files, relative to the notebooks/ folder

## 1. Data for crossdating

Read a collection and detrend it to ring-width indices (RWI) — crossdating works
on the detrended series.

In [ ]:
rwl = dpl.readers(RWL + "ca533.rwl")
rwi = dpl.detrend(rwl, fit="Spline")
rwi.iloc[:5, :4]

## 2. Mean interseries correlation

How strongly each series agrees with the master built from all the others (the
statistic COFECHA reports as the interseries correlation). This one takes the
**raw** ring widths — it normalizes internally.

In [ ]:
mean_corr, ic = dpl.interseries_corr(rwl)
mean_corr                              # collection-wide mean interseries correlation

In [ ]:
ic.head()                             # the per-series table

## 3. Crossdating with `xdate`

Correlate each series against the leave-one-out master over overlapping segments.
Segments that fail are flagged **A** (not significant) or **B** (correlates
better at a non-zero lag — a possible dating shift). The flag report prints as it
runs; the full results come back in a dict.

In [ ]:
res = dpl.xdate(rwi)
list(res.keys())

In [ ]:
res["overall"].head()                  # per-series correlation with the master

## 4. The crossdating overview plot

Green = a series' extent, blue = a segment that dates well, red = a flagged
segment to inspect.

In [ ]:
dpl.xdate_plot(rwi)
plt.show()

## 5. Detecting a dating error

Introduce a deliberate mistake — shift one series five years off its true dates —
and crossdating catches it: every segment flags, and the best correlation sits at
lag -5, the size of the error.

In [ ]:
bad = rwi.copy()
col = bad["CAM041"].dropna()
bad["CAM041"] = pd.Series(col.values, index=col.index + 5).reindex(bad.index)

res_bad = dpl.xdate(bad)

In [ ]:
res_bad["flags"]["CAM041"]["B"][0]     # best lag = -5, the shift we introduced

In [ ]:
dpl.xdate_plot(bad)                    # CAM041 now reads red
plt.show()

## 6. Drilling into one series

`series_corr` examines a single series against the master: a moving correlation
and a per-segment lag panel. For the misdated series the peak sits off zero, at
the -5 shift.

In [ ]:
sc = dpl.series_corr(bad, "CAM041", make_plot=True)
plt.show()

## 7. Dating a floating series

Given a dated reference collection, `xdate_floater` finds the calendar placement
of an undated series. Here we take a series, discard its dates, and recover
them.

In [ ]:
floating = rwl["CAM021"].dropna().values      # just the ring values, undated
reference = rwl.drop(columns=["CAM021"])

fl = dpl.xdate_floater(reference, floating, series_name="CAM021", make_plot=True)
plt.show()

In [ ]:
fl["best"]                             # recovered span + crossdating statistics

## 8. COFECHA emulation

`preset="COFECHA"` reproduces the COFECHA program (Burg prewhitening, a 32-year
spline, an arithmetic z-scored master, Pearson correlation, and COFECHA's segment
anchoring and critical value), returning its "possible problems" count.

In [ ]:
rwi32 = dpl.detrend(rwl, fit="Spline", period=32)
cof = dpl.xdate(rwi32, preset="COFECHA")
cof["n_problems"]

## 9. Prewhitening under the hood

Crossdating prewhitens each series with an autoregressive model first, to compare
the high-frequency signal. `autoreg` exposes that fit directly.

In [ ]:
dpl.autoreg(rwl["CAM011"].dropna())    # selected AR order and coefficients

## 10. A batch QA report

`xdate_report` reads, detrends, crossdates, and produces a COFECHA-style report
per file — built for batch QA. `output=` controls where it goes: `"screen"`
prints it here, `"file"` writes a `.txt`, and `"both"` (the default) does both.

In [ ]:
rep = dpl.xdate_report(RWL + "ca533.rwl", output="screen")